In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

In [ ]:
df = pd.read_csv("FIT VSPP 2565-2567.csv")
fuel_types = df['เชื้อเพลิง'].dropna().unique()
print(fuel_types)

In [ ]:
import pandas as pd
df = pd.read_csv("FIT VSPP 2565-2567.csv")
df_sorted = df.sort_values(by='id')
df = df[df['year'] != 2568]
df = df.drop(columns=['kwhp', 'kwhop','year','เขต'])
print(df_sorted.head())

In [ ]:
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)
df_sorted = df.sort_values(by='id')
df = df[df['year'] != 2568]
df = df.drop(columns=['kwhp', 'kwhop','year','เขต'])
print(df_sorted.head())

In [ ]:
import numpy as np
import pandas as pd

df['kWH'] = df['kWH'].astype(str).str.replace(',', '').str.strip()
df['kWH'] = df['kWH'].replace(['-', '', ' ', 'nan', '<NA>'], np.nan)
df['kWH'] = df['kWH'].astype(float).fillna(0)
df['AVG_kWH'] = df.groupby(['id', 'month'])['kWH'].transform('mean')
df['kWH_zero'] = (df['kWH'] == 0).astype(int)
missing_rate_df = df.groupby('id')['kWH_zero'].mean().reset_index(name='missing_rate_kWH')
missing_rate_df['missing_rate_kWH'] = (missing_rate_df['missing_rate_kWH'] * 100).round(2)
aggregated_df = df.groupby(['id', 'month'], as_index=False).apply(lambda g: g.iloc[0]).reset_index(drop=True)
aggregated_df = aggregated_df.fillna(0)
aggregated_df['AVG_kWH'] = aggregated_df['AVG_kWH'].round(2)
aggregated_df = aggregated_df.merge(missing_rate_df, on='id', how='left')
print(aggregated_df.drop(columns=['kWH', 'kWH_zero']).head())

In [ ]:
days_per_month = {
    1: 31, 2: 28, 3: 31, 4: 30,
    5: 31, 6: 30, 7: 31, 8: 31,
    9: 30, 10: 31, 11: 30, 12: 31
}
aggregated_df['PF100'] = aggregated_df['month'].map(days_per_month) * aggregated_df['mwตามสัญญา'] * 24000
print(aggregated_df[['id', 'month', 'mwตามสัญญา', 'PF100']].head())

In [ ]:
aggregated_df['AVG_kWH'] = aggregated_df['AVG_kWH'].replace(0, np.nan)
aggregated_df['efficiency'] = (aggregated_df['AVG_kWH']/aggregated_df['PF100'] ) * 100
aggregated_df['efficiency'] = aggregated_df['efficiency'].fillna(0)
aggregated_df['efficiency'] = aggregated_df['efficiency'].round(2)
print(aggregated_df[['id', 'month', 'PF100', 'AVG_kWH', 'efficiency']].head())

In [ ]:
aggregated_df.drop(columns=['kWH','kWH_zero']).to_csv('1.csv', index=False)